# Eval Log Parser & Player Profiler
This notebook parses the evaluation `.txt` logs from `data/eval_logs/`, extracts player profiling metrics using tile information from the board layout, and outputs profiles in the same format as `player_profiler.ipynb`.

In [1]:
import polars as pl
from pathlib import Path
import json
import re

# Directory containing the .txt logs
logs_dir = Path('../../data/eval_logs')
txt_files = list(logs_dir.glob('*.txt'))

print(f"Found {len(txt_files)} log files to parse")


Found 500 log files to parse


In [2]:
# Catanatron node-to-coordinate mapping for starting resource extraction
# Uses Board() to get the standard map geometry
from catanatron.models.board import Board

b = Board()
cm = b.map

# Build tile_id -> coord mapping
tile_id_to_coord = {}
for coord, tile in cm.land_tiles.items():
    tile_id_to_coord[tile.id] = coord

# Build node_id -> list of coord tuples mapping
node_tile_map = {}
for node_id, tiles in cm.adjacent_tiles.items():
    node_tile_map[node_id] = [tile_id_to_coord[t.id] for t in tiles]

print(f"Built node-to-tile mapping for {len(node_tile_map)} nodes")


Built node-to-tile mapping for 54 nodes


In [3]:
node_tile_map


{0: [(0, 0, 0), (0, 1, -1), (1, 0, -1)],
 1: [(0, 0, 0), (1, -1, 0), (1, 0, -1)],
 2: [(0, 0, 0), (1, -1, 0), (0, -1, 1)],
 3: [(0, 0, 0), (0, -1, 1), (-1, 0, 1)],
 4: [(0, 0, 0), (-1, 0, 1), (-1, 1, 0)],
 5: [(0, 0, 0), (-1, 1, 0), (0, 1, -1)],
 6: [(1, -1, 0), (1, 0, -1), (2, -1, -1)],
 7: [(1, -1, 0), (2, -2, 0), (2, -1, -1)],
 8: [(1, -1, 0), (2, -2, 0), (1, -2, 1)],
 9: [(1, -1, 0), (0, -1, 1), (1, -2, 1)],
 10: [(0, -1, 1), (1, -2, 1), (0, -2, 2)],
 11: [(0, -1, 1), (0, -2, 2), (-1, -1, 2)],
 12: [(0, -1, 1), (-1, 0, 1), (-1, -1, 2)],
 13: [(-1, 0, 1), (-1, -1, 2), (-2, 0, 2)],
 14: [(-1, 0, 1), (-2, 0, 2), (-2, 1, 1)],
 15: [(-1, 0, 1), (-1, 1, 0), (-2, 1, 1)],
 16: [(-1, 1, 0), (0, 1, -1), (-1, 2, -1)],
 17: [(-1, 1, 0), (-2, 1, 1), (-2, 2, 0)],
 18: [(-1, 1, 0), (-2, 2, 0), (-1, 2, -1)],
 19: [(0, 1, -1), (0, 2, -2), (1, 1, -2)],
 20: [(0, 1, -1), (1, 0, -1), (1, 1, -2)],
 21: [(0, 1, -1), (-1, 2, -1), (0, 2, -2)],
 22: [(1, 0, -1), (1, 1, -2), (2, 0, -2)],
 23: [(1, 0, -1), (

In [4]:
def parse_txt_log(filepath, node_tile_map):
    """
    Parse a single .txt log file and extract all relevant metrics for P0 (our bot).
    Uses tile information from the board layout section.

    Returns a dictionary with all the metrics needed for profiling.
    """
    with open(filepath, 'r') as f:
        content = f.read()

    metrics = {}

    # Parse header
    header_match = re.search(r'BOT_NAME:\s*(\w+)', content)
    if header_match:
        metrics['bot_name'] = header_match.group(1)

    game_id_match = re.search(r'GAME_ID:\s*(\d+)', content)
    if game_id_match:
        metrics['game_id'] = int(game_id_match.group(1))

    winner_match = re.search(r'WINNER:\s*Color\.(\w+)', content)
    if winner_match:
        metrics['winner'] = winner_match.group(1)

    p0_color_match = re.search(r'P0_COLOR:\s*Color\.(\w+)', content)
    if p0_color_match:
        metrics['p0_color'] = p0_color_match.group(1)

    # Parse board layout into a coordinate -> {resource, number} dict
    board_section = re.search(r'--- BOARD LAYOUT ---\n(.*?)--- ACTIONS ---', content, re.DOTALL)
    board_map = {}
    if board_section:
        board_lines = board_section.group(1).strip().split('\n')
        for line in board_lines:
            match = re.search(r'HEX\s*\(([^)]+)\):\s*(\w+)\s*(\d+)', line)
            if match:
                coord_str = match.group(1)
                resource = match.group(2)
                number = int(match.group(3))
                coord = tuple(map(int, coord_str.split(', ')))
                board_map[coord] = {'resource': resource, 'number': number}

    # Parse actions
    actions_section = re.search(r'--- ACTIONS ---\n(.*?)--- FINAL PLAYER STATE', content, re.DOTALL)
    actions = []
    if actions_section:
        actions_text = actions_section.group(1)
        actions = [line.strip() for line in actions_text.split('\n') if line.strip() and not line.startswith('---')]

    bot_color = metrics.get('p0_color', '').upper()
    bot_actions = [a for a in actions if f'[{bot_color}]' in a]

    # Initialize counters
    trades_proposed = 0
    trades_completed = 0
    counter_offers = 0
    bank_trades = 0
    port_used = 0
    cards_given_in_trades = 0
    cards_taken_in_trades = 0
    turns_before_first_trade = -1
    first_trade_seen = False

    traded_away = {'ore': 0, 'wool': 0, 'lumber': 0, 'grain': 0, 'brick': 0}
    received_trade = {'ore': 0, 'wool': 0, 'lumber': 0, 'grain': 0, 'brick': 0}

    roads_built = 0
    settlements_built = 0
    cities_built = 0
    dev_cards_bought = 0
    knights_played = 0

    players_targeted = 0
    times_targeted = 0
    unique_players_stolen_from = set()

    settlement_coords = []

    for i, action in enumerate(actions):
        if f'[{bot_color}]' not in action:
            # Track if OTHER players target us
            if 'MOVE_ROBBER' in action and f'<Color.{bot_color}' in action:
                times_targeted += 1
            continue

        # Maritime trades (bank/port trades)
        if 'MARITIME_TRADE' in action:
            trades_proposed += 1
            trades_completed += 1  # Maritime trades always complete
            bank_trades += 1
            if not first_trade_seen:
                turns_before_first_trade = i
                first_trade_seen = True

            match = re.search(r"\('(.*)'\)", action)
            if match:
                resources_str = match.group(1)
                resources = [r.strip().strip("'") for r in resources_str.split(', ')]
                # Format can be (given, given, given, given, received) for 4:1
                # or (given, given, None, None, received) for port trades (3:1 or 2:1)
                if len(resources) == 5:
                    given_resource = resources[0]
                    received_resource = resources[4]
                    
                    # Check if it's a port trade (has None values)
                    if 'None' in resources:
                        port_used += 1
                        # Count actual given items (non-None)
                        given_count = sum(1 for r in resources[:4] if r != 'None')
                    else:
                        given_count = 4

                    if given_resource and given_resource != 'None':
                        res_key = given_resource.lower()
                        if res_key in traded_away:
                            traded_away[res_key] += given_count
                            cards_given_in_trades += given_count
                    
                    if received_resource and received_resource != 'None':
                        res_key = received_resource.lower()
                        if res_key in received_trade:
                            received_trade[res_key] += 1
                            cards_taken_in_trades += 1

        elif 'BUILD_ROAD' in action:
            roads_built += 1

        elif 'BUILD_SETTLEMENT' in action:
            settlements_built += 1
            match = re.search(r'\|\s*BUILD_SETTLEMENT\s*\|\s*(\d+)', action)
            if match:
                node_id = int(match.group(1))
                settlement_coords.append(node_id)

        elif 'BUILD_CITY' in action:
            cities_built += 1

        elif 'BUY_DEVELOPMENT_CARD' in action:
            dev_cards_bought += 1

        elif 'PLAY_KNIGHT_CARD' in action:
            knights_played += 1

        elif 'MOVE_ROBBER' in action:
            # Check if targeting a specific player
            target_match = re.search(r'<Color\.(\w+):', action)
            if target_match:
                target_color = target_match.group(1)
                if target_color != bot_color:
                    players_targeted += 1
                    unique_players_stolen_from.add(target_color)

    # Parse final player state JSON
    json_match = re.search(r'--- FINAL PLAYER STATE \(P0\) ---\n(\{.*\})', content, re.DOTALL)
    player_state = {}
    if json_match:
        try:
            player_state = json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass

    # Extract stats from final state
    hand_size = (
        player_state.get('P0_WOOD_IN_HAND', 0) +
        player_state.get('P0_BRICK_IN_HAND', 0) +
        player_state.get('P0_SHEEP_IN_HAND', 0) +
        player_state.get('P0_WHEAT_IN_HAND', 0) +
        player_state.get('P0_ORE_IN_HAND', 0)
    )

    longest_road_length = player_state.get('P0_LONGEST_ROAD_LENGTH', 0)
    has_largest_army = player_state.get('P0_HAS_ARMY', False)
    actual_vp = player_state.get('P0_ACTUAL_VICTORY_POINTS', 0)

    # Calculate starting resources from settlement coordinates + board map
    starting_ore = 0
    starting_wool = 0
    starting_lumber = 0
    starting_grain = 0
    starting_brick = 0

    # Resource name mapping: Catanatron uses these names
    resource_map = {
        'WOOD': 'lumber',
        'WHEAT': 'grain',
        'ORE': 'ore',
        'SHEEP': 'wool',
        'BRICK': 'brick',
        'DESERT': None
    }

    for node_id in settlement_coords[:2]:  # First two settlements for starting resources
        if node_id in node_tile_map:
            adjacent_tiles = node_tile_map[node_id]
            for tile in adjacent_tiles:
                # tile is a coordinate tuple
                if tile in board_map:
                    res = board_map[tile]['resource']
                    mapped = resource_map.get(res)
                    if mapped == 'ore':
                        starting_ore += 1
                    elif mapped == 'wool':
                        starting_wool += 1
                    elif mapped == 'lumber':
                        starting_lumber += 1
                    elif mapped == 'grain':
                        starting_grain += 1
                    elif mapped == 'brick':
                        starting_brick += 1

    # Did we win?
    won = metrics.get('winner', '').upper() == bot_color

    # Build final metrics dict
    metrics.update({
        'turns_before_first_trade': turns_before_first_trade,
        'cards_given_in_trades': cards_given_in_trades,
        'cards_gained_in_trades': cards_taken_in_trades,
        'trades_proposed': trades_proposed,
        'trades_completed': trades_completed,
        'counter_offers': counter_offers,
        'bank_trades': bank_trades,
        'port_used': 1 if port_used > 0 else 0,
        'dev_cards_bought': dev_cards_bought,
        'roads_built': roads_built,
        'settlements_built': settlements_built,
        'cities_built': cities_built,
        'unique_players_stolen_from': len(unique_players_stolen_from),
        'times_targeted_by_others': times_targeted,
        'largest_army_received': 1 if has_largest_army else 0,
        'longest_road_received': 1 if longest_road_length >= 5 else 0,
        'avg_hand_size': hand_size,
        'won': 1 if won else 0,
        'actual_vp': actual_vp,
        'longest_road_length': longest_road_length,
        'starting_ore': starting_ore,
        'starting_wool': starting_wool,
        'starting_lumber': starting_lumber,
        'starting_grain': starting_grain,
        'starting_brick': starting_brick,
        'traded_away_ore': traded_away['ore'],
        'traded_away_wool': traded_away['wool'],
        'traded_away_lumber': traded_away['lumber'],
        'traded_away_grain': traded_away['grain'],
        'traded_away_brick': traded_away['brick'],
        'received_trade_ore': received_trade['ore'],
        'received_trade_wool': received_trade['wool'],
        'received_trade_lumber': received_trade['lumber'],
        'received_trade_grain': received_trade['grain'],
        'received_trade_brick': received_trade['brick'],
    })

    return metrics

print("parse_txt_log function defined successfully")


parse_txt_log function defined successfully


In [5]:
# Parse all log files
all_metrics = []
for txt_file in txt_files:
    try:
        metrics = parse_txt_log(txt_file, node_tile_map)
        all_metrics.append(metrics)
        if len(all_metrics) % 50 == 0:
            print(f"Parsed {len(all_metrics)} files...")
    except Exception as e:
        print(f"Error parsing {txt_file}: {e}")

if all_metrics:
    df_all = pl.DataFrame(all_metrics)
    print(f"Aggregated {len(all_metrics)} games.")
else:
    print("No metrics parsed!")


Parsed 50 files...
Parsed 100 files...
Parsed 150 files...
Parsed 200 files...
Parsed 250 files...
Parsed 300 files...
Parsed 350 files...
Parsed 400 files...
Parsed 450 files...
Parsed 500 files...
Aggregated 500 games.


In [6]:
if 'df_all' in locals() and not df_all.is_empty():
    # Calculate player profiles (aggregated per bot)
    player_profiles = df_all.group_by('bot_name').agg(
        num_games=pl.len(),
        avg_turns_before_first_trade=pl.col('turns_before_first_trade').filter(
            pl.col('turns_before_first_trade') != -1).mean(),
        ratio_cards_given_to_taken=(
            pl.col('cards_given_in_trades').sum() / 
            pl.col('cards_gained_in_trades').sum().clip(lower_bound=1)
        ),
        trade_success_rate=(
            pl.col('trades_completed').sum() / 
            pl.col('trades_proposed').sum().clip(lower_bound=1)
        ),
        avg_counter_offers=pl.col('counter_offers').mean(),
        avg_bank_trades=pl.col('bank_trades').mean(),
        avg_dev_cards_bought=pl.col('dev_cards_bought').mean(),
        avg_roads_built=pl.col('roads_built').mean(),
        avg_cities_built=pl.col('cities_built').mean(),
        avg_players_targeted=pl.col('unique_players_stolen_from').mean(),
        avg_times_targeted=pl.col('times_targeted_by_others').mean(),
        win_rate_largest_army=(pl.col('largest_army_received') > 0).sum() / pl.len(),
        win_rate_longest_road=(pl.col('longest_road_received') > 0).sum() / pl.len(),
        average_games_with_port_used=pl.col('port_used').sum() / pl.len(),
        overall_avg_hand_size=pl.col('avg_hand_size').mean(),
        
        # Starting resources totals
        total_starting_ore=pl.col('starting_ore').sum(),
        total_starting_wool=pl.col('starting_wool').sum(),
        total_starting_lumber=pl.col('starting_lumber').sum(),
        total_starting_grain=pl.col('starting_grain').sum(),
        total_starting_brick=pl.col('starting_brick').sum(),
        
        # Trading behavior totals
        total_traded_away_ore=pl.col('traded_away_ore').sum(),
        total_traded_away_wool=pl.col('traded_away_wool').sum(),
        total_traded_away_lumber=pl.col('traded_away_lumber').sum(),
        total_traded_away_grain=pl.col('traded_away_grain').sum(),
        total_traded_away_brick=pl.col('traded_away_brick').sum(),
        
        total_received_trade_ore=pl.col('received_trade_ore').sum(),
        total_received_trade_wool=pl.col('received_trade_wool').sum(),
        total_received_trade_lumber=pl.col('received_trade_lumber').sum(),
        total_received_trade_grain=pl.col('received_trade_grain').sum(),
        total_received_trade_brick=pl.col('received_trade_brick').sum()
    )
    
    print("Player profiles aggregated successfully")
else:
    print("Could not generate player profiles because df_all is empty.")


Player profiles aggregated successfully


In [7]:
if 'player_profiles' in locals():
    # Calculate top 3 starting resources and trading behaviors
    derived_metrics_list = []
    
    for row in player_profiles.iter_rows(named=True):
        bot_name = row['bot_name']
        
        # Starting resources
        res_counts = {
            'ore': row['total_starting_ore'],
            'wool': row['total_starting_wool'],
            'lumber': row['total_starting_lumber'],
            'grain': row['total_starting_grain'],
            'brick': row['total_starting_brick']
        }
        sorted_res = sorted(res_counts.items(), key=lambda item: item[1], reverse=True)
        top_3 = [res for res, count in sorted_res[:3]]
        
        # Traded away stats
        traded_away = {
            'ore': row['total_traded_away_ore'],
            'wool': row['total_traded_away_wool'],
            'lumber': row['total_traded_away_lumber'],
            'grain': row['total_traded_away_grain'],
            'brick': row['total_traded_away_brick']
        }
        most_traded_away = max(traded_away, key=traded_away.get) if sum(traded_away.values()) > 0 else "None"
        
        # Received stats
        received_trade = {
            'ore': row['total_received_trade_ore'],
            'wool': row['total_received_trade_wool'],
            'lumber': row['total_received_trade_lumber'],
            'grain': row['total_received_trade_grain'],
            'brick': row['total_received_trade_brick']
        }
        most_received = max(received_trade, key=received_trade.get) if sum(received_trade.values()) > 0 else "None"
        
        derived_metrics_list.append({
            'bot_name': bot_name,
            'top_3_starting_resources': ", ".join(top_3),
            'most_traded_away_resource': most_traded_away,
            'most_received_resource': most_received
        })
    
    df_derived = pl.DataFrame(derived_metrics_list)
    
    # Join the derived columns back
    player_profiles = player_profiles.join(df_derived, on='bot_name', how='left')
    
    # Drop the intermediate aggregate columns to clean up
    columns_to_drop = [
        'total_starting_ore', 'total_starting_wool', 'total_starting_lumber',
        'total_starting_grain', 'total_starting_brick',
        'total_traded_away_ore', 'total_traded_away_wool', 'total_traded_away_lumber',
        'total_traded_away_grain', 'total_traded_away_brick',
        'total_received_trade_ore', 'total_received_trade_wool', 'total_received_trade_lumber',
        'total_received_trade_grain', 'total_received_trade_brick'
    ]
    player_profiles = player_profiles.drop(columns_to_drop)
    
    # Sort by number of games played (descending)
    player_profiles = player_profiles.sort('num_games', descending=True)
    
    # Display the result
    print("PLAYER PROFILES:")
    display(player_profiles)
else:
    print("No player profiles to display.")


PLAYER PROFILES:


bot_name,num_games,avg_turns_before_first_trade,ratio_cards_given_to_taken,trade_success_rate,avg_counter_offers,avg_bank_trades,avg_dev_cards_bought,avg_roads_built,avg_cities_built,avg_players_targeted,avg_times_targeted,win_rate_largest_army,win_rate_longest_road,average_games_with_port_used,overall_avg_hand_size,top_3_starting_resources,most_traded_away_resource,most_received_resource
str,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str
"""aware_axelrod""",500,54.987976,2.5531,1.0,0.0,39.724,7.95,9.818,0.636,0.874,8.124,0.48,0.67,0.728,3.834,"""grain, wool, lumber""","""ore""","""ore"""


In [8]:
# Save the filtered player profiles dataframe to parquet
output_dir = Path('../../data/player_profiles')
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'parsed_eval_logs_profiles.parquet'
player_profiles.write_parquet(output_path)

print(f"Player profiles exported successfully to {output_path.resolve()}")


Player profiles exported successfully to /Users/aatmiya/Documents/GitHub/Redeem-Catan/data/player_profiles/parsed_eval_logs_profiles.parquet


In [ ]:
# Optional: Clean up the .txt logs (commented out for safety)
# Uncomment the lines below if you want to delete the txt files after parsing

# import shutil
# for txt_file in txt_files:
#     txt_file.unlink()
# print(f"Cleaned up {len(txt_files)} txt files from {logs_dir}")
